# SurgiDoc: U-Net fine-tuning on CholecSeg8k

Before running:
1. **Add data** → search `newslab/cholecseg8k` → add it.
2. **Settings** → Accelerator: **GPU T4 x2** (or P100), Internet: **on**.
3. Optional: add a Kaggle secret `WANDB_API_KEY` to log the run to Weights & Biases.

Outputs land in `/kaggle/working/runs/`: `best.pt`, `history.json`, `results.json`.
Download them from the Output tab, or commit the notebook ("Save Version") to keep them.

In [ ]:
!git clone -q https://github.com/Muthusaicharan77/SurgiDocAi.git
%cd SurgiDocAi
!git log --oneline -1
!pip install -q -e ".[train]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path
# Locate the folder that contains video01/, wherever Kaggle mounted the dataset.
ROOT = next(p.parent for p in Path("/kaggle/input").rglob("video01") if p.is_dir())
print(ROOT, sorted(d.name for d in ROOT.iterdir())[:5])

In [ ]:
import os
WANDB = ""
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    WANDB = "--wandb"
except Exception:
    print("No WANDB_API_KEY secret; training without W&B logging.")

Smoke test first (about a minute): catches path or install problems before the long run.

In [ ]:
!python -m surgidoc.vision.train --root {ROOT} --out /kaggle/working/runs/smoke \
    --epochs 1 --limit-batches 5 --workers 4

Full run. ResNet34 encoder with ImageNet weights, 256x448 input, 30 epochs.

In [ ]:
!python -m surgidoc.vision.train --root {ROOT} --out /kaggle/working/runs/unet_r34 \
    --encoder resnet34 --epochs 30 --batch-size 16 --workers 4 {WANDB}

In [ ]:
import json
r = json.load(open("/kaggle/working/runs/unet_r34/results.json"))
print("best epoch", r["best_epoch"], "| test mIoU", r["test"]["mean_iou"], "| test mDice", r["test"]["mean_dice"])
for name, s in r["test"]["per_class"].items():
    print(f"{name:24s} {s['frames']:5d}  " + (f"dice {s['dice']:.3f}  iou {s['iou']:.3f}" if s["dice"] is not None else "not scored"))